In [3]:
import seaborn as sns
import numpy as np
import glob
import matplotlib.pyplot as plt
import pprint
import matplotlib.pyplot as plt
from itertools import combinations
from  pathlib import Path
from readerscf import parse_sdr_file
from get_project_root import get_project_root
from replace_outliers import replace_outliers
from calculate_inverse_matrix import calculate_inverse_matrix
from detrend import detrend_dataframe,detrend_preserve_mean_level,detrend_and_shift_positive,detrend_preserve_baseline,detrend_full
from substractdf import subtract_column_min,subtract_percentile_norm
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe,matrix_multiply_with_dataframe
from subtract_mean_from_first_n import subtract_mean_from_first_n
from remove_baseline import remove_baseline
from config import IUPAC, ref_str, color_map
from deconvolve_nnls import deconvolve_nnls
from center_dataframe import center_dataframe
from estimate_M_from_data import estimate_M_from_data
from evaluate_quality import evaluate_quality
from estimate_crosstalk_matrix import estimate_crosstalk_matrix
from estimate_M_yin import estimate_M_yin
from estimate_M_correlation import estimate_M_correlation,estimate_M_correlation_crostalk
from deconvolve_domnisoru import deconvolve_domnisoru
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe
from estimate_M_goodpeaks_crostalk import estimate_M_goodpeaks_crostalk
from estimate_M_clusters_crostalk import estimate_M_clusters_crostalk
from estimate_M_from_clean_peaks import estimate_M_from_clean_peaks
from estimate_M_sklearn import estimate_M_sklearn
from  divide_matrices import  divide_matrices
from compare_matrices import compare_matrices
from replace_outliers import replace_outliers
from normalize_diagonal import normalize_diagonal
from divide_matrices_np import divide_matrices_np
from tqdm import tqdm 

def main():
    project_root = get_project_root()
    project_root = Path(get_project_root())
    print(f"📁 Корень проекта: {project_root}")
    


    # 🔹 Точный путь к папке с файлами
    srd_dir = project_root / "files" / "сырые SRD" / "сырые SRD" 

    if not srd_dir.is_dir():
        print(f"❌ Папка не найдена: {srd_dir}")
        return

    srd_files = sorted(srd_dir.glob("*.srd"))
    if not srd_files:
        print("⚠️ Файлы .srd не найдены в папке tqmd.")
        return

    print(f"📂 Найдено файлов: {len(srd_files)}\n")
    # 🔹 Цикл с прогресс-баром
    with tqdm(srd_files, desc="🔄 Обработка SRD", unit="файл") as pbar:
        for srd_path in pbar:
            # Отображаем текущий файл справа в строке прогресса
            pbar.set_postfix(file=srd_path.name[:20] + "..." if len(srd_path.name) > 18 else srd_path.name)
            
            try:
                sdr_matrix, sdr_channels, sdr_meta = parse_sdr_file(srd_path)
                
                matrix = sdr_matrix.to_numpy()
                if matrix.shape[0] < 10 or matrix.shape[1] < 4:
                    tqdm.write(f"   ⚠️ Матрица слишком мала в {srd_path.name}, пропуск.")
                    continue
                    
                matrixdef = matrix[6:10, 0:4]
                tqdm.write(f"   matrixdef:\n{matrixdef}")
                tqdm.write(f"   matrix inv:\n{np.linalg.inv(matrixdef)}")

                data0 = sdr_channels.loc[:, ['dR110', 'dR6G', 'dTAMRA', 'dROX']]
                data0.columns = ['G', 'A', 'T', 'C']
                
                data0 = subtract_column_min(data0)
                data0 = center_dataframe(data0, method='percentile', percentile=2)
                
                matrix1 = estimate_M_from_data(
                    raw=data0,
                    dye_order=['G', 'A', 'T', 'C'],
                    min_purity=0.8,
                    peak_height=400,
                    peak_distance=20,
                    peak_prominence=150
                )
                tqdm.write(f"   matrix1:\n{matrix1}")
                matrix2 = estimate_crosstalk_matrix(data0, init_M=matrixdef)
                tqdm.write(f"   matrix2:\n{matrix2}")
                
                matrix1def = divide_matrices_np(matrix1, matrixdef)
                tqdm.write(f"   matrix1def:\n{matrix1def}")
                matrix2def = divide_matrices_np(matrix2, matrixdef)
                tqdm.write(f"   matrix2def:\n{matrix2def}")

                tqdm.write(f"   ✅ {srd_path.name} готов")
                
            except Exception as e:
                tqdm.write(f"   ❌ Ошибка в {srd_path.name}: {e}")
    
if __name__ == "__main__":
    main()

📁 Корень проекта: C:\Users\Admin\Documents\GitHub\dnasegnercrosstalk
📂 Найдено файлов: 40



🔄 Обработка SRD:   2%|▊                                | 1/40 [00:00<00:12,  3.06файл/s, file=3_pGEM_A3_PDMA6_50.s...]

   matrixdef:
[[1.         0.45442271 0.05485368 0.00475959]
 [0.73666734 1.         0.39780498 0.01566364]
 [0.43882051 0.62656242 1.         0.25624526]
 [0.20345469 0.31454486 0.56733644 1.        ]]
   matrix inv:
[[ 1.51032801 -0.84882143  0.29411056 -0.06925735]
 [-1.1277791   1.97319603 -0.82913169  0.1869214 ]
 [ 0.03709389 -0.87648276  1.5667097  -0.38790957]
 [ 0.02640909  0.04929865 -0.68789057  1.1753708 ]]
Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
   matrix1:
[[ 1.          0.26980728  0.10088707  0.        ]
 [ 0.          1.15631692 -0.16151205  0.        ]
 [ 0.         -0.39614561  1.22521667  0.        ]
 [ 0.         -0.02997859 -0.16459169  1.        ]]
Найдено пиков: 592
  Итерация 1: max Δ = 0.625512
  Итерация 2: max Δ = 0.026133
  Итерация 3: max Δ = 0.010547
  Сходимость на итерации 5
   matrix2:
[[ 0.36

🔄 Обработка SRD:   5%|█▋                               | 2/40 [00:01<00:20,  1.88файл/s, file=3_pGEM_B3_PDMA6_50.s...]

   matrixdef:
[[1.         0.43628675 0.05150481 0.00714676]
 [0.7464748  1.         0.38239858 0.02149237]
 [0.4566119  0.6320703  1.         0.29575685]
 [0.22059479 0.33302453 0.55968916 1.        ]]
   matrix inv:
[[ 1.48598492 -0.7931545   0.26743835 -0.07266993]
 [-1.11648009  1.91864401 -0.78800983  0.19980231]
 [ 0.01696632 -0.85482712  1.57680796 -0.44810075]
 [ 0.03451886  0.01444771 -0.67909123  1.20028867]]
Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 1 (A) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 2 (T) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
   matrix1:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
Найдено пиков: 871
  Итерация 1: max Δ = 0.619864
  Итерация 2: max Δ = 0.010523
  Итерация 3: max Δ

...

   matrixdef:
[[1.         0.41367722 0.05000864 0.00458785]
 [0.81624645 1.         0.39529222 0.01520487]
 [0.50024062 0.64153248 1.         0.24483733]
 [0.23944218 0.33585784 0.58281732 1.        ]]
   matrix inv:
[[ 1.51607232 -0.77816412  0.26705034 -0.06050752]
 [-1.2522847   1.98904799 -0.82742163  0.17808579]
 [ 0.0360235  -0.89679739  1.5686896  -0.37060336]
 [ 0.03658284  0.04095701 -0.70030654  1.1706706 ]]
Предупреждение: для канала 0 (G) не найдено чистых пиков. Использован единичный вектор как fallback.
   matrix1:
[[ 1.          0.1607565  -0.11507937  0.08768971]
 [ 0.          0.94562648 -0.14880952 -0.0623946 ]
 [ 0.         -0.13947991  0.98412698  0.0623946 ]
 [ 0.          0.03309693  0.2797619   0.91231029]]
Найдено пиков: 534
  Итерация 1: max Δ = 0.636632
  Итерация 2: max Δ = 0.017081
  Итерация 3: max Δ = 0.008422
  Итерация 6: max Δ = 0.001524
  Сходимость на итерации 8
   matrix2:
[[0.3332066  0.17760334 0.06696619 0.04146645]
 [0.26479243 0.40389787 0.1711

🔄 Обработка SRD:  98%|███████████████████████████████▏| 39/40 [00:23<00:00,  1.97файл/s, file=pGEM_4_H2_PDMA6_36.s...]

   matrixdef:
[[1.         0.44193712 0.05277209 0.0041219 ]
 [0.7750569  1.         0.39160573 0.01317801]
 [0.48771745 0.65593278 1.         0.25148645]
 [0.2322792  0.33857206 0.57360756 1.        ]]
   matrix inv:
[[ 1.52691042 -0.84034155  0.28719093 -0.06744437]
 [-1.19639401  2.01093008 -0.83200395  0.18766908]
 [ 0.03199465 -0.91972572  1.5794473  -0.38522132]
 [ 0.03204368  0.04191076 -0.69099811  1.17309228]]
Предупреждение: для канала 1 (A) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 2 (T) не найдено чистых пиков. Использован единичный вектор как fallback.
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
   matrix1:
[[ 1.10045662  0.          0.          0.        ]
 [ 0.06621005  1.          0.          0.        ]
 [-0.10273973  0.          1.          0.        ]
 [-0.06392694  0.          0.          1.        ]]
Найдено пиков: 545
  Итерация 1: max Δ = 0.638572
  Итера

🔄 Обработка SRD: 100%|████████████████████████████████| 40/40 [00:24<00:00,  1.64файл/s, file=pGEM_4_H2_PDMA6_36.s...]

   matrixdef:
[[1.         0.40660971 0.04700698 0.00482402]
 [0.81167859 1.         0.40013742 0.0151031 ]
 [0.51497883 0.65486789 1.         0.26874182]
 [0.24013181 0.33527166 0.55782586 1.        ]]
   matrix inv:
[[ 1.4960869  -0.76474481  0.27439427 -0.06940835]
 [-1.22279812  1.9861538  -0.85145426  0.20472309]
 [ 0.01963306 -0.91430102  1.59662497 -0.41536583]
 [ 0.03975967  0.02775924 -0.67106101  1.1797311 ]]
Предупреждение: для канала 3 (C) не найдено чистых пиков. Использован единичный вектор как fallback.
   matrix1:
[[ 1.11827957 -0.0576123  -0.01774722  0.        ]
 [ 0.26075269  1.83210733 -0.15049014  0.        ]
 [ 0.0188172  -0.66101899  0.89037529  0.        ]
 [-0.39784946 -0.11347603  0.27786207  1.        ]]
Найдено пиков: 525
  Итерация 1: max Δ = 0.647206
  Итерация 2: max Δ = 0.014774
  Итерация 3: max Δ = 0.011243
  Итерация 6: max Δ = 0.000897
  Сходимость на итерации 7
   matrix2:
[[0.3328091  0.17104585 0.06874238 0.05497331]
 [0.28225309 0.4154683  0.1707